# 5.8.3 Model Serving and Deployment

REST API patterns, batch inference, latency, ONNX, TF Serving, Triton.

In [ ]:
import numpy as np, time
# Minimal prediction handler
class Handler:
    def __init__(self, F=4): self.F=F; self.W=np.random.randn(F+1)*0.4
    def predict(self, X):
        Xb=np.column_stack([np.ones(len(X)),X])
        return (1/(1+np.exp(-(Xb@self.W).clip(-50,50)))>=0.5).astype(int)
    def handle(self, payload):
        X=np.atleast_2d(payload['instances'])
        return {'predictions':self.predict(X).tolist()}, 200
h=Handler(4)
resp,status=h.handle({'instances':np.random.randn(3,4).tolist()})
print(resp, status)

In [ ]:
# Latency benchmark
batch_sizes=[1,8,32,128]
times={bs:[] for bs in batch_sizes}
for _ in range(30):
    for bs in batch_sizes:
        t0=time.perf_counter(); h.handle({'instances':np.random.randn(bs,4).tolist()})
        times[bs].append((time.perf_counter()-t0)*1000)
import matplotlib.pyplot as plt
plt.plot(batch_sizes,[np.mean(times[bs]) for bs in batch_sizes],'o-')
plt.xlabel('Batch size'); plt.ylabel('ms'); plt.title('Serving Latency'); plt.show()